In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [2]:
file_path = "../data/raw/it_support_tickets_raw.csv"

df = pd.read_csv(file_path)

df.head()

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
0,TCKT_000001,2024-01-31T05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back ...,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,0,android,EU
1,TCKT_000002,2024-10-20T06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,0,web,NaN
2,TCKT_000003,2024-06-18T21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my changes.,Thanks for reporting this bug. We will look into this and get back to you within the next 48 hours.,NaN,NaN,0,neutral,3,0,android,MEA
3,TCKT_000004,2025-12-25T15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. Our team is reviewing this and will ...,NaN,NaN,0,positive,5,1,android,LATAM
4,TCKT_000005,2023-08-27T16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you with...,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,0,web,NaN


In [3]:
# Check dataset size
df.shape

(10000, 20)

In [4]:
# Check column names
df.columns

Index(['ticket_id', 'created_at', 'customer_id', 'customer_segment', 'channel',
       'product_area', 'issue_type', 'priority', 'status', 'sla_plan',
       'initial_message', 'agent_first_reply', 'resolution_summary',
       'resolution_time_hours', 'reopened', 'customer_sentiment', 'csat_score',
       'has_attachment', 'platform', 'region'],
      dtype='object')

In [5]:
# Basic dataset information
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ticket_id              10000 non-null  object 
 1   created_at             10000 non-null  object 
 2   customer_id            10000 non-null  object 
 3   customer_segment       10000 non-null  object 
 4   channel                10000 non-null  object 
 5   product_area           10000 non-null  object 
 6   issue_type             10000 non-null  object 
 7   priority               10000 non-null  object 
 8   status                 10000 non-null  object 
 9   sla_plan               10000 non-null  object 
 10  initial_message        10000 non-null  object 
 11  agent_first_reply      10000 non-null  object 
 12  resolution_summary     5954 non-null   object 
 13  resolution_time_hours  5954 non-null   float64
 14  reopened               10000 non-null  int64  
 15  cus

## Data Understanding

The dataset contains 10,000 synthetic IT support ticket records and 20 columns. Each row represents one support ticket raised by a customer or user.

The dataset includes ticket-level information such as ticket ID, creation date, customer segment, support channel, product area, issue type, ticket priority, status, SLA plan, resolution time, customer sentiment, CSAT score, platform, and region.

This dataset is suitable for analyzing IT support operations, SLA performance, ticket resolution trends, customer experience, product-area risk, and operational bottlenecks.

In [6]:
# Check missing values by column
missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values

resolution_time_hours    4046
resolution_summary       4046
region                   1989
created_at                  0
platform                    0
has_attachment              0
csat_score                  0
customer_sentiment          0
reopened                    0
agent_first_reply           0
ticket_id                   0
sla_plan                    0
status                      0
priority                    0
issue_type                  0
product_area                0
channel                     0
customer_segment            0
customer_id                 0
initial_message             0
dtype: int64

In [7]:
# Check missing value percentage
missing_percentage = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

missing_percentage

resolution_time_hours    40.46
resolution_summary       40.46
region                   19.89
created_at                0.00
platform                  0.00
has_attachment            0.00
csat_score                0.00
customer_sentiment        0.00
reopened                  0.00
agent_first_reply         0.00
ticket_id                 0.00
sla_plan                  0.00
status                    0.00
priority                  0.00
issue_type                0.00
product_area              0.00
channel                   0.00
customer_segment          0.00
customer_id               0.00
initial_message           0.00
dtype: float64

## Missing Values Assessment

The missing value analysis shows that most columns are complete. Only three columns contain missing values: `resolution_time_hours`, `resolution_summary`, and `region`.

The missing values in `resolution_time_hours` and `resolution_summary` likely represent tickets that have not yet been resolved. This is expected in an IT support dataset because open, in-progress, or on-hold tickets may not have final resolution details.

The `region` column has 19.89% missing values. For this project, missing regions will be labeled as `Unknown` instead of removing the records, because the tickets can still provide useful insight for SLA, priority, channel, product-area, and status analysis.

In [8]:
# Check ticket status distribution
df["status"].value_counts()

status
resolved            5014
in_progress         2020
on_hold             1029
open                 997
closed_no_action     940
Name: count, dtype: int64

In [9]:
# Check missing resolution time by ticket status
df.groupby("status")["resolution_time_hours"].apply(lambda x: x.isnull().sum()).sort_values(ascending=False)

status
in_progress         2020
on_hold             1029
open                 997
closed_no_action       0
resolved               0
Name: resolution_time_hours, dtype: int64

## Missing Resolution Fields by Ticket Status

The missing `resolution_time_hours` values are linked to ticket status rather than random data quality issues.

Tickets with the status `open`, `in_progress`, and `on_hold` do not have resolution times because they have not reached a final resolution stage. In contrast, tickets marked as `resolved` and `closed_no_action` have complete resolution time values.

This confirms that the missing resolution fields should not be removed or filled with artificial averages. Instead, these records should be preserved and treated as active or unresolved tickets in the analysis.

In [10]:
# Create a working copy of the dataset for cleaning
df_clean = df.copy()

# Convert created_at to datetime
df_clean["created_at"] = pd.to_datetime(df_clean["created_at"], errors="coerce")

# Fill missing region values
df_clean["region"] = df_clean["region"].fillna("Unknown")

# Fill missing resolution summary for unresolved tickets
df_clean["resolution_summary"] = df_clean["resolution_summary"].fillna("Not yet resolved")

# Create ticket age fields from created_at
df_clean["created_year"] = df_clean["created_at"].dt.year
df_clean["created_month"] = df_clean["created_at"].dt.month
df_clean["created_month_name"] = df_clean["created_at"].dt.month_name()
df_clean["created_quarter"] = df_clean["created_at"].dt.quarter

# Create resolved flag
df_clean["is_resolved"] = df_clean["status"].isin(["resolved", "closed_no_action"])

# Create unresolved flag
df_clean["is_unresolved"] = df_clean["status"].isin(["open", "in_progress", "on_hold"])

# Preview cleaned dataset
df_clean.head()

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region,created_year,created_month,created_month_name,created_quarter,is_resolved,is_unresolved
0,TCKT_000001,2024-01-31 05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back ...,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,0,android,EU,2024,1,January,1,True,False
1,TCKT_000002,2024-10-20 06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,0,web,Unknown,2024,10,October,4,True,False
2,TCKT_000003,2024-06-18 21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my changes.,Thanks for reporting this bug. We will look into this and get back to you within the next 48 hours.,Not yet resolved,NaN,0,neutral,3,0,android,MEA,2024,6,June,2,False,True
3,TCKT_000004,2025-12-25 15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. Our team is reviewing this and will ...,Not yet resolved,NaN,0,positive,5,1,android,LATAM,2025,12,December,4,False,True
4,TCKT_000005,2023-08-27 16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you with...,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,0,web,Unknown,2023,8,August,3,True,False


In [11]:
# Check cleaned dataset info
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ticket_id              10000 non-null  object        
 1   created_at             10000 non-null  datetime64[ns]
 2   customer_id            10000 non-null  object        
 3   customer_segment       10000 non-null  object        
 4   channel                10000 non-null  object        
 5   product_area           10000 non-null  object        
 6   issue_type             10000 non-null  object        
 7   priority               10000 non-null  object        
 8   status                 10000 non-null  object        
 9   sla_plan               10000 non-null  object        
 10  initial_message        10000 non-null  object        
 11  agent_first_reply      10000 non-null  object        
 12  resolution_summary     10000 non-null  object        
 13  re

In [12]:
# Confirm missing values after cleaning
df_clean.isnull().sum().sort_values(ascending=False)

resolution_time_hours    4046
created_at                  0
is_resolved                 0
created_quarter             0
created_month_name          0
created_month               0
created_year                0
region                      0
platform                    0
has_attachment              0
csat_score                  0
customer_sentiment          0
reopened                    0
ticket_id                   0
resolution_summary          0
agent_first_reply           0
initial_message             0
sla_plan                    0
status                      0
priority                    0
issue_type                  0
product_area                0
channel                     0
customer_segment            0
customer_id                 0
is_unresolved               0
dtype: int64

In [13]:
# Save cleaned dataset to processed folder
output_path = "../data/processed/it_support_tickets_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print(output_path)

Cleaned dataset saved successfully.
../data/processed/it_support_tickets_cleaned.csv
